In [1]:
# ============================================
# Ollama + Qwen3-8B-Q4_K_M.gguf RAG 完全版
# ============================================

import os
import psycopg2
from pgvector.psycopg2 import register_vector
import fitz  # PyMuPDF
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
import ollama


# ============================================
# 1. PostgreSQL 接続
# ============================================
conn = psycopg2.connect(
    host="192.168.11.56",
    dbname="ragdb",
    user="postgres",
    password="hirorian77",
)
register_vector(conn)
cur = conn.cursor()


# ============================================
# 2. テーブル & index 初期化（768次元）
# ============================================
def init_table():
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    cur.execute("DROP TABLE IF EXISTS chunks_ollama;")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS chunks_ollama (
            id SERIAL PRIMARY KEY,
            source    TEXT NOT NULL,
            text      TEXT NOT NULL,
            embedding vector(768) NOT NULL
        );
    """)
    conn.commit()


def init_index():
    try:
        cur.execute("""
            CREATE INDEX chunks_ollama_embedding_hnsw
            ON chunks_ollama
            USING hnsw (embedding vector_cosine_ops);
        """)
        conn.commit()
    except Exception:
        conn.rollback()


# ============================================
# 3. PDF → テキスト
# ============================================
def load_pdf_text(path: str) -> str:
    doc = fitz.open(path)
    texts = []
    for page in doc:
        texts.append(page.get_text("text"))
    return "\n".join(texts)


# ============================================
# 3-2. TXT → テキスト
# ============================================
def load_txt_text(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


# ============================================
# 3-3. PDF/TXT 自動判定
# ============================================
def load_document_text(path: str) -> str:
    if path.lower().endswith(".pdf"):
        return load_pdf_text(path)
    elif path.lower().endswith(".txt"):
        return load_txt_text(path)
    else:
        return ""


# ============================================
# 4. チャンク分割
# ============================================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=80,
    separators=["\n\n", "\n", "。", "、", " "],
)

def chunk_text(text: str):
    return text_splitter.split_text(text)


# ============================================
# 5. Embedding（Ollama nomic-embed-text）
# ============================================
def embed_passages(texts):
    vecs = []
    for t in texts:
        out = ollama.embeddings(
            model="nomic-embed-text",
            prompt=f"passage: {t}"
        )
        vecs.append(out["embedding"])
    return np.array(vecs, dtype=np.float32)


def embed_query(query: str):
    out = ollama.embeddings(
        model="nomic-embed-text",
        prompt=f"query: {query}"
    )
    return np.array(out["embedding"], dtype=np.float32)


# ============================================
# 6. PostgreSQL 保存
# ============================================
def save_to_postgres(source, chunks, embeddings):
    for text, emb in zip(chunks, embeddings):
        safe_text = str(text)
        safe_emb = [float(x) for x in emb.tolist()]
        cur.execute(
            "INSERT INTO chunks_ollama (source, text, embedding) VALUES (%s, %s, %s)",
            (str(source), safe_text, safe_emb),
        )
    conn.commit()


# ============================================
# 7. PDF/TXT フォルダ → pgvector 構築
# ============================================
def build_pgvector_from_docs(doc_dir: str):
    for filename in os.listdir(doc_dir):
        if not (filename.lower().endswith(".pdf") or filename.lower().endswith(".txt")):
            continue

        path = os.path.join(doc_dir, filename)
        print(f"[DOC] {path}")

        text = load_document_text(path)
        if not text.strip():
            print(f"テキストなし: {path}")
            continue

        chunks = chunk_text(text)
        if not chunks:
            print(f"チャンクなし: {path}")
            continue

        chunks_with_source = [
            f"[source: {filename}]\n{c}" for c in chunks
        ]

        embs = embed_passages(chunks_with_source)
        save_to_postgres(filename, chunks_with_source, embs)
        print(f"保存完了: {len(chunks_with_source)} チャンク ({filename})")


# ============================================
# 8. 追加 DOC（PDF/TXT）追記
# ============================================
def append_docs_to_postgres(doc_dir: str):
    for filename in os.listdir(doc_dir):
        if not (filename.lower().endswith(".pdf") or filename.lower().endswith(".txt")):
            continue

        path = os.path.join(doc_dir, filename)
        print(f"[追加DOC] {path}")

        text = load_document_text(path)
        if not text.strip():
            print(f"テキストなし: {path}")
            continue

        chunks = chunk_text(text)
        if not chunks:
            print(f"新しいチャンクなし: {path}")
            continue

        chunks_with_source = [
            f"[source: {filename}]\n{c}" for c in chunks
        ]

        embs = embed_passages(chunks_with_source)
        save_to_postgres(filename, chunks_with_source, embs)
        print(f"追加保存完了: {len(chunks_with_source)} チャンク ({filename})")


# ============================================
# 9. pgvector 検索（スコア付き）
# ============================================
def search_pgvector(query: str, top_k: int = 3):
    q_emb = embed_query(query).tolist()

    cur.execute("""
        SELECT source, text, (embedding <-> (%s::vector)) AS distance
        FROM chunks_ollama
        ORDER BY embedding <-> (%s::vector)
        LIMIT %s;
    """, (q_emb, q_emb, top_k))

    rows = cur.fetchall()

    results = []
    for source, text, dist in rows:
        score = 1.0 - float(dist)
        results.append((source, text, score))
    return results


# ============================================
# 10. Ollama LLM（Qwen3-8B-Q4_K_M.gguf）
# ============================================
def generate_answer(prompt: str) -> str:
    out = ollama.chat(
        model="Qwen3-8B-Q4_K_M",   # ★ここを固定
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return out["message"]["content"].strip()


# ============================================
# 11. プロンプト構築
# ============================================
def build_prompt(query: str, contexts):
    ctx_blocks = []
    for source, text, score in contexts:
        ctx_blocks.append(
            f"[source={source}, score={score:.4f}]\n{text}"
        )
    context_text = "\n\n".join(ctx_blocks)

    prompt = f"""
あなたは与えられた情報のみを使って、日本語で簡潔かつ正確に回答してください。
与えられた情報にない内容は推測せず、「分かりません」と答えてください。

【検索結果】
{context_text}

【質問】
{query}

【回答】
"""
    return prompt


# ============================================
# 12. RAG パイプライン
# ============================================
def rag_pg(query: str, top_k: int = 3) -> str:
    contexts = search_pgvector(query, top_k=top_k)
    prompt = build_prompt(query, contexts)
    answer = generate_answer(prompt)
    return answer


C:\Users\USER\miniforge3\envs\rag_test\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'ollama'

In [9]:

# ============================================
# 14. main
# ============================================
if __name__ == "__main__":
    PDF_DIR = "docs"  # PDF を置くフォルダ

 #   print("=== テーブル初期化（768次元, nomic-embed-text） ===")
 #   init_table()
 #   init_index()

 #   print("=== PDF → pgvector 構築 ===")
 #   build_pgvector_from_docs(PDF_DIR)


In [2]:
PDF_DIR = "docs"  # PDF を置くフォルダ
print("=== PDF → pgvector 追加 ===")
append_docs_to_postgres(PDF_DIR)

=== PDF → pgvector 追加 ===
[追加DOC] docs\soseiheptares.blogspot.com_2026_05_263.html.txt


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]


追加保存完了: 2 チャンク (soseiheptares.blogspot.com_2026_05_263.html.txt)
[追加DOC] docs\soseiheptares.blogspot.com_search_updated-max_2025-11-06T08_30_00+09_00_max-results_13_start_9_by-date_false.txt


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.19s/it]


追加保存完了: 42 チャンク (soseiheptares.blogspot.com_search_updated-max_2025-11-06T08_30_00+09_00_max-results_13_start_9_by-date_false.txt)
[追加DOC] docs\soseiheptares.blogspot.com_search_updated-max_2026-01-13T08_48_00+09_00_max-results_13.txt


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.46s/it]


追加保存完了: 54 チャンク (soseiheptares.blogspot.com_search_updated-max_2026-01-13T08_48_00+09_00_max-results_13.txt)
[追加DOC] docs\soseiheptares.blogspot.com_search_updated-max_2026-05-12T18_42_00+09_00_max-results_13_reverse-paginate_true.txt


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.47s/it]


追加保存完了: 52 チャンク (soseiheptares.blogspot.com_search_updated-max_2026-05-12T18_42_00+09_00_max-results_13_reverse-paginate_true.txt)


In [8]:
print("\n=== RAG 実行 ===")
q = "Direclidineの効果"
ans = rag_pg(q, top_k=3)
print("\n【質問】", q)
print("【回答】\n", ans)


=== RAG 実行 ===

【質問】 Direclidineの効果
【回答】
 分かりません。
